In [40]:
# Setup: imports and environment checks
import os
import sys
from pathlib import Path

# Ensure project paths
ROOT = Path(r"c:\Users\junhongs\Desktop\capstone\evaluation")
MATERIAL_DIR = ROOT / "material"
OUTPUT_DIR = ROOT / "dataset" / "uas_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find all PDF files in material directory
pdf_files = list(MATERIAL_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files in material directory:")
for pdf in pdf_files:
    size_kb = pdf.stat().st_size / 1024
    print(f"  - {pdf.name} ({size_kb:.1f} KB)")

# Soft dependency checks
missing = []
for pkg in ["ragas", "langchain_community", "langchain_openai", "openai", "pypdf", "pandas"]:
    try:
        __import__(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    print("Missing packages detected:\n - " + "\n - ".join(missing))
    print("Install them in this kernel, for example:")
    print("%pip install ragas langchain-community langchain-openai openai pypdf pandas tqdm")
else:
    print("All required packages found.")

Found 46 PDF files in material directory:
  - AccessMatri-Aws-Key-Management-Service-Implementation-Guide-6.0.1-GA (AI).pdf (436.6 KB)
  - AccessMatrix-AI-Deployment-Guide-6.0.1-GA (AI).pdf (192.1 KB)
  - AccessMatrix-Azure-Key-Vault-HSM-Implementation-Guide-6.0.1-GA (AI).pdf (358.3 KB)
  - AccessMatrix-Cloud-Deployment-Guide-6.0.1-GA (AI).pdf (2911.1 KB)
  - AccessMatrix-Common-Administration-Guide-6.0.1-GA (AI).pdf (4194.7 KB)
  - AccessMatrix-Common-Deployment-Guide-6.0.1-GA (AI).pdf (927.3 KB)
  - AccessMatrix-Common-Hardening-Guide-6.0.1-GA (AI).pdf (409.6 KB)
  - AccessMatrix-Common-Login-Page-Deployment-Guide-6.0 (AI).pdf (3365.4 KB)
  - AccessMatrix-Google-Cloud-HSM-Implementation-Guide-6.0.1-GA (AI).pdf (336.4 KB)
  - AccessMatrix-Hashicorp-Vault-Implementation-Guide-6.0.1-GA (AI).pdf (405.2 KB)
  - AccessMatrix-Performance-Benchmark-Report-6.0.1-GA (AI).pdf (462.9 KB)
  - AccessMatrix-Product-Upgrade-Guide-6.0.1-GA (AI).pdf (319.6 KB)
  - AccessMatrix-Supported-Platforms-6.0.

In [41]:
# Shared utility functions
import numpy as np
import pandas as pd

def _is_nonempty_value(x):
    """Check if a value is non-empty (handles strings, lists, arrays, NaN)"""
    if x is None:
        return False
    if isinstance(x, str):
        return x.strip() != ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) > 0
    if isinstance(x, np.ndarray):
        return x.size > 0
    try:
        import math
        if isinstance(x, float) and math.isnan(x):
            return False
    except Exception:
        pass
    return True

def infer_columns(row):
    """Extract query, ground_truth, and contexts from a RAGAS DataFrame row"""
    # Query
    query = None
    for k in ["user_input", "question", "query", "prompt"]:
        if k in row and pd.notna(row[k]):
            query = row[k]
            break
    
    # Ground truth
    gt = None
    for k in ["reference", "ground_truth", "expected_output", "answer"]:
        if k in row:
            val = row[k]
            if _is_nonempty_value(val):
                gt = val
                break
    
    # Contexts
    contexts = None
    for k in ["contexts", "reference_contexts", "contexts_text", "documents"]:
        if k in row:
            val = row[k]
            if not _is_nonempty_value(val):
                continue
            # Ensure list[str]
            if isinstance(val, str):
                contexts = [val]
            elif isinstance(val, (list, tuple)):
                first = val[0] if len(val) else None
                if isinstance(first, dict) and "page_content" in first:
                    contexts = [d.get("page_content", "") for d in val]
                else:
                    contexts = [str(v) for v in val]
            elif isinstance(val, np.ndarray):
                if val.size: 
                    first = val.flat[0]
                    if isinstance(first, dict) and "page_content" in first:
                        contexts = [d.get("page_content", "") for d in val.tolist()]
                    else:
                        contexts = [str(v) for v in val.tolist()]
            elif isinstance(val, dict) and "page_content" in val:
                contexts = [val.get("page_content", "")]            
            else:
                contexts = [str(val)]
            break
    
    return query, gt, contexts or []

In [ ]:
# Process all PDFs in material directory
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Storage for all processed documents
all_pdf_data = []

for pdf_path in pdf_files:
    print(f"\n{'='*60}")
    print(f"Processing: {pdf_path.name}")
    print(f"{'='*60}")
    
    # Determine question count based on file size
    size_kb = pdf_path.stat().st_size / 1024
    testset_size = 15 if size_kb > 1000 else 10
    
    # Load PDF
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    total_chars = sum(len(d.page_content) for d in docs)
    
    print(f" Pages: {len(docs)} | Characters: {total_chars:,}")
    print(f" Will generate: {testset_size} questions (size: {size_kb:.1f} KB)")
    
    # Check content quality
    if total_chars < 1000:
        print(f" WARNING: Very little text extracted! Skipping this PDF.")
        continue
    
    # Chunk the document
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2048,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", ".", " "]
    )
    chunked_docs = text_splitter.split_documents(docs)
    
    chunk_sizes = [len(d.page_content) for d in chunked_docs]
    print(f"Chunks: {len(chunked_docs)} | Avg size: {sum(chunk_sizes)//len(chunk_sizes)} chars")
    
    # Store for later processing
    all_pdf_data.append({
        'pdf_path': pdf_path,
        'pdf_name': pdf_path.stem,
        'chunks': chunked_docs,
        'testset_size': testset_size,
        'total_chars': total_chars
    })

print(f"\n Successfully loaded {len(all_pdf_data)} PDFs for processing")


Processing: AccessMatri-Aws-Key-Management-Service-Implementation-Guide-6.0.1-GA (AI).pdf
📄 Pages: 25 | Characters: 34,212
📊 Will generate: 10 questions (size: 436.6 KB)
📑 Chunks: 28 | Avg size: 1243 chars

Processing: AccessMatrix-AI-Deployment-Guide-6.0.1-GA (AI).pdf
📄 Pages: 9 | Characters: 6,974
📊 Will generate: 10 questions (size: 192.1 KB)
📑 Chunks: 8 | Avg size: 871 chars

Processing: AccessMatrix-Azure-Key-Vault-HSM-Implementation-Guide-6.0.1-GA (AI).pdf
📄 Pages: 25 | Characters: 34,212
📊 Will generate: 10 questions (size: 436.6 KB)
📑 Chunks: 28 | Avg size: 1243 chars

Processing: AccessMatrix-AI-Deployment-Guide-6.0.1-GA (AI).pdf
📄 Pages: 9 | Characters: 6,974
📊 Will generate: 10 questions (size: 192.1 KB)
📑 Chunks: 8 | Avg size: 871 chars

Processing: AccessMatrix-Azure-Key-Vault-HSM-Implementation-Guide-6.0.1-GA (AI).pdf
📄 Pages: 21 | Characters: 23,818
📊 Will generate: 10 questions (size: 358.3 KB)
📑 Chunks: 19 | Avg size: 1263 chars

Processing: AccessMatrix-Cloud-Deploym

In [ ]:
# Verify API key is visible to this kernel and optionally load from .env
import os

# Optional: auto-load from a .env file if present
try:
    from dotenv import load_dotenv  # type: ignore
    loaded = load_dotenv()
    if loaded:
        print("Loaded environment from .env")
except Exception:
    pass  # python-dotenv not installed; that's okay

api_key = os.environ.get("OPENAI_API_KEY", "")
masked = (api_key[:4] + "***" + api_key[-4:]) if api_key else None
print("OPENAI_API_KEY set:", bool(api_key), f"({masked})" if masked else "(None)")



Loaded environment from .env
OPENAI_API_KEY set: True (sk-s***mWMA)


In [44]:
domain_prompt = """
### 1. ROLE AND GOAL
You are an Expert-level Software Engineer and Test Set Generator at 'i-sprint innovations.' Your goal is to create a "golden" evaluation dataset for a new RAG system. This dataset will test the RAG's ability to provide accurate, grounded technical support to developers and deployment engineers working with the i-sprint product suite (like UAS).

### 2. TASK
You will be provided with a technical document chunk. Your task is to generate 3-5 high-quality, complex question-answer pairs based *exclusively* on this text. You must output a valid JSON list.

### 3. GENERATION RULES
- **100% GROUNDED:** The `question` must be answerable *only* with the provided text. The `ground_truth_answer` must be a concise, factual summary of the answer. The `source_context` must be the *exact quote(s)* from the text that support the answer.
- **PERSONA-DRIVEN:** Questions must sound like they are from a technical engineer. Use professional, specific, and concise phrasing.
- **TOPIC FOCUS:** Questions must target key identity and security concepts: authentication flows, identity federation (SAML, OIDC), authorization policies, OATH token implementation, security configurations, deployment steps, or troubleshooting error codes.
- **DIVERSE SCENARIOS:** You must generate questions from at least two of the following `question_type` categories:
    1.  **Deployment/Configuration:** Questions about setup, high-availability, or setting parameters.
    2.  **Troubleshooting/Error Handling:** Questions about resolving errors, log analysis, or audit procedures.
    3.  **Integration/Development:** Questions about API endpoints, coding standards, or integrating with protocols like OAuth2.

### 4. OUTPUT FORMAT
Respond ONLY with a valid JSON list. Do not include any text before or after the JSON.

[
  {
    "question_type": "Deployment/Configuration | Troubleshooting/Error Handling | Integration/Development",
    "question": "A specific, professional question based *only* on the document.",
    "ground_truth_answer": "A concise, factual answer derived *only* from the document.",
    "source_context": "The exact quote or passage from the document that contains the answer."
  },
  {
    "question_type": "...",
    "question": "...",
    "ground_truth_answer": "...",
    "source_context": "..."
  }
]

"""


In [45]:
# Import Persona class and create persona objects
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="DevOps Engineer",
        role_description="Troubleshoots production deployments, version upgrades, and compatibility issues. Asks scenario-based questions about migration paths and configuration conflicts."
    ),
    Persona(
        name="Security Architect", 
        role_description="Evaluates authentication protocols, authorization flows, and compliance requirements. Asks about OAuth/SAML implementations and security implications."
    )
]



In [ ]:
# RAGAS setup and batch generation for all PDFs
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# Setup generator once
generator_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4o', temperature=0.1))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    persona_list=personas
)

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.6), 
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.2)
]

# Generate questions for each PDF
generated_datasets = []

for pdf_data in all_pdf_data:
    print(f"\n  Generating questions for: {pdf_data['pdf_name']}")
    print(f"   Target: {pdf_data['testset_size']} questions")
    
    try:
        dataset = generator.generate_with_langchain_docs(
            pdf_data['chunks'],
            testset_size=pdf_data['testset_size'],
            query_distribution=query_distribution
        )
        
        df = dataset.to_pandas()
        
        # Filter out MISSPELLED questions if present
        if 'query_style' in df.columns:
            before = len(df)
            df = df[df['query_style'] != 'MISSPELLED'].copy()
            removed = before - len(df)
            if removed > 0:
                print(f"    Removed {removed} misspelled questions")
        
        generated_datasets.append({
            'pdf_name': pdf_data['pdf_name'],
            'df': df,
            'testset_size': pdf_data['testset_size']
        })
        
        print(f"   Generated {len(df)} questions")
        
    except Exception as e:
        print(f"    Error: {e}")
        continue

print(f"\n🎉 Successfully generated datasets for {len(generated_datasets)} PDFs")

C:\Users\junhongs\AppData\Local\Temp\ipykernel_41804\2938811417.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4o', temperature=0.1))
C:\Users\junhongs\AppData\Local\Temp\ipykernel_41804\2938811417.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))
C:\Users\junhongs\AppData\Local\Temp\ipykernel_41804\2938811417.py:14: DeprecationWarning: LangchainEmbedding


🔄 Generating questions for: AccessMatri-Aws-Key-Management-Service-Implementation-Guide-6.0.1-GA (AI)
   Target: 10 questions


Applying CustomNodeFilter:   0%|          | 0/28 [00:00<?, ?it/s]Node 9c6f221b-c1f6-4d08-889c-b2166498b8b4 does not have a summary. Skipping filtering.
Node 1bdb5341-015c-416f-87a0-c07a31cb6ab5 does not have a summary. Skipping filtering.
Node 8e762f36-efa2-44cf-ad28-05abc55c5336 does not have a summary. Skipping filtering.
Node 131f8ca4-9ff4-4862-a769-4ea209efbb66 does not have a summary. Skipping filtering.
Node f3f5c3f7-fce0-4d65-ad2d-27c4a8d185f8 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:   0%|          | 0/28 [00:00<?, ?it/s]Node 9c6f221b-c1f6-4d08-889c-b2166498b8b4 does not have a summary. Skipping filtering.
Node 1bdb5341-015c-416f-87a0-c07a31cb6ab5 does not have a summary. Skipping filtering.
Node 8e762f36-efa2-44cf-ad28-05abc55c5336 does not have a summary. Skipping filtering.
Node 131f8ca4-9ff4-4862-a769-4ea209efbb66 does not have a summary. Skipping filtering.
Node f3f5c3f7-fce0-4d65-ad2d-27c4a8d185f8 does not have a summary. Skipping filtering.


   ✅ Generated 10 questions

🔄 Generating questions for: AccessMatrix-AI-Deployment-Guide-6.0.1-GA (AI)
   Target: 10 questions


Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]Node ea27a65a-f1db-4985-8287-6338174adf53 does not have a summary. Skipping filtering.
Node 6d70da26-98f8-4e84-a6e1-9c300d0849c8 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]Node ea27a65a-f1db-4985-8287-6338174adf53 does not have a summary. Skipping filtering.
Node 6d70da26-98f8-4e84-a6e1-9c300d0849c8 does not have a summary. Skipping filtering.
Generating Samples: 100%|██████████| 10/10 [00:19<00:00,  1.91s/it]



...

Applying CustomNodeFilter:   0%|          | 0/92 [00:00<?, ?it/s]Node c4e23281-b22b-4ece-88ab-dd329bf5cfb7 does not have a summary. Skipping filtering.
Node 029f2c3f-6055-4640-8090-979493ab4e95 does not have a summary. Skipping filtering.
Node 4d1fca65-7e5c-4ddc-9c19-ab7ce84814e6 does not have a summary. Skipping filtering.
Node 0773851e-8de2-4576-ba66-1f846b9f0d18 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:   0%|          | 0/92 [00:00<?, ?it/s]Node c4e23281-b22b-4ece-88ab-dd329bf5cfb7 does not have a summary. Skipping filtering.
Node 029f2c3f-6055-4640-8090-979493ab4e95 does not have a summary. Skipping filtering.
Node 4d1fca65-7e5c-4ddc-9c19-ab7ce84814e6 does not have a summary. Skipping filtering.
Node 0773851e-8de2-4576-ba66-1f846b9f0d18 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  47%|████▋     | 43/92 [01:07<01:06,  1.35s/it]Node bea1d7a7-fbd5-4bb8-8169-4068e39aa1af does not have a summary. Skipping filtering.
Node bea1d7a7

   ⚠️ Removed 1 misspelled questions
   ✅ Generated 14 questions

🔄 Generating questions for: UAS-OneSpan-DPMobileGateway-Implementation-Guide-6.0.1-GA (AI)
   Target: 10 questions


Applying CustomNodeFilter:   0%|          | 0/33 [00:00<?, ?it/s]Node 9ea05f97-2537-4664-b1da-8945361880fe does not have a summary. Skipping filtering.
Node 18d61b8d-9a0d-4ad7-9948-50df59316b3a does not have a summary. Skipping filtering.
Node 88d900ff-6603-491f-8f83-5de5f7eafc4a does not have a summary. Skipping filtering.
Node e1e6f728-76f5-48a9-b8a4-73d63c067163 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:   0%|          | 0/33 [00:00<?, ?it/s]Node 9ea05f97-2537-4664-b1da-8945361880fe does not have a summary. Skipping filtering.
Node 18d61b8d-9a0d-4ad7-9948-50df59316b3a does not have a summary. Skipping filtering.
Node 88d900ff-6603-491f-8f83-5de5f7eafc4a does not have a summary. Skipping filtering.
Node e1e6f728-76f5-48a9-b8a4-73d63c067163 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:  15%|█▌        | 5/33 [00:02<00:14,  1.89it/s]Node 2bd58f58-0304-4173-9427-97793d230cda does not have a summary. Skipping filtering.
Node 78feb1e5-

   ⚠️ Removed 2 misspelled questions
   ✅ Generated 8 questions

🔄 Generating questions for: UAS-OneSpan-Multi-Device-Implementation-Guide-6.0.1-GA (AI)
   Target: 15 questions


Applying SummaryExtractor:  97%|█████████▋| 65/67 [02:25<00:03,  1.83s/it]

In [ ]:
# Preview generated datasets
import pandas as pd

print(" Generated Datasets Summary:\n")
for data in generated_datasets:
    print(f" {data['pdf_name']}")
    print(f"   Questions: {len(data['df'])} / {data['testset_size']} requested")
    print(f"   Columns: {list(data['df'].columns)}")
    print()

# Show sample from first dataset
if generated_datasets:
    print("Sample from first dataset:")
    display(generated_datasets[0]['df'][['user_input', 'synthesizer_name']].head(3))

In [ ]:
# Save all datasets to individual JSONL and CSV files
import json
import pandas as pd

for data in generated_datasets:
    pdf_name = data['pdf_name']
    df = data['df']
    
    # Create output directory for this PDF
    output_subdir = OUTPUT_DIR / pdf_name
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    jsonl_path = output_subdir / "test.jsonl"
    csv_path = output_subdir / "test.csv"
    
    # Convert DataFrame rows to standardized format
    records = []
    for _, row in df.iterrows():
        q, gt, ctx = infer_columns(row)
        records.append({
            "query": q,
            "ground_truth": gt,
            "contexts": ctx,
        })
    
    # Write JSONL
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    
    # Write CSV
    pd.DataFrame({
        "query": [r["query"] for r in records],
        "ground_truth": [r["ground_truth"] for r in records],
        "contexts_joined": ["\n\n".join(r["contexts"]) for r in records]
    }).to_csv(csv_path, index=False, encoding="utf-8")
    
    print(f" {pdf_name}")
    print(f"   └─ {output_subdir.relative_to(ROOT)}")
    print(f"      ├─ {jsonl_path.name} ({len(records)} questions)")
    print(f"      └─ {csv_path.name}")

print(f"\n🎉 All datasets saved to: {OUTPUT_DIR.relative_to(ROOT)}")

Columns: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']
Saved to: dataset\uas_dataset\AccessMatrix-Supported-Platforms-6.0.1-GA (AI)
 - test2.jsonl
 - test2.csv
